 ## Conjunto do Compressor de Reciclo do HDT IV REPLAN

- Modelagem do conjunto Motor, acoplamento de baixa velocidade, conjunto de transmissão, acoplamento de alta velocidade e compressor
- Análise da vibração torcional e comparação com Documento já feito (I-RL-5270.00-52313-321-BI4-104=B)

Importação de bibliotecas

In [ ]:
import sys
# sys.path.append('C:\\Users\\Murillo\\OneDrive - Universidade Federal de Uberlândia\\Área de Trabalho\\Mestrado\\ENGRENAMENTO\\Implementacao\\ross')
sys.path.append('C:\\Users\\Murillo\\OneDrive - Universidade Federal de Uberlândia\\Área de Trabalho\\Mestrado\\ENGRENAMENTO\\Implementacao\\ross_dev_backlash\\ross')

#home
# sys.path.append('C:\\Users\\M\\Documents\\Mestrado\\ENGRENAMENTO\\ross')

from backlash import Backlash

import ross as rs
import numpy as np
import scipy as sp
import copy
import plotly.graph_objects as go
# from sklearn.preprocessing import MinMaxScaler
from ross.units import Q_
from ross.probe import Probe
from tabulate import tabulate
import pandas as pd
from ross.utils import convert_6dof_to_torsional

import matplotlib.pyplot as plt

In [ ]:
def concatenate_rotor(rotor_list):
    """Concatena múltiplos objetos ross.Rotor em um único rotor,
    realocando nós e garantindo unicidade dos tags.
    """
    from copy import deepcopy
    shaft_elements = []
    disk_elements = []
    bearing_elements = []
    point_mass_elements = []

    node_offset = 0
    rotor_id = 0  # Identificador incremental para tags

    for rotor in rotor_list:
        rotor = deepcopy(rotor)

        # Reindexar elementos de eixo
        for i, el in enumerate(rotor.shaft_elements):
            el.n_l += node_offset
            el.n_r += node_offset
            el.n = el.n_l  # importante para elementos do ROSS
            el.tag = f"shaft_r{rotor_id}_{i}"
        shaft_elements.extend(rotor.shaft_elements)

        # Reindexar discos
        for i, el in enumerate(rotor.disk_elements):
            el.n += node_offset
            el.tag = f"disk_r{rotor_id}_{i}"
        disk_elements.extend(rotor.disk_elements)

        # Reindexar mancais
        for i, el in enumerate(rotor.bearing_elements):
            el.n += node_offset
            el.tag = f"bearing_r{rotor_id}_{i}"
        bearing_elements.extend(rotor.bearing_elements)

        # Reindexar massas concentradas
        for i, el in enumerate(rotor.point_mass_elements):
            el.n += node_offset
            el.tag = f"pmass_r{rotor_id}_{i}"
        point_mass_elements.extend(rotor.point_mass_elements)

        # Atualiza deslocamento para o próximo rotor
        all_nodes = [el.n_r for el in rotor.shaft_elements] + [el.n for el in rotor.disk_elements + rotor.bearing_elements]
        node_offset = max(all_nodes) #+ 1
        rotor_id += 1

    rotor_concat = rs.Rotor(
        shaft_elements=shaft_elements,
        disk_elements=disk_elements,
        bearing_elements=bearing_elements,
        point_mass_elements=point_mass_elements,
    )

    return rotor_concat

Definição de Material

In [ ]:
steel = rs.Material(name="Steel", rho=7850, E=2e11, Poisson=0.3)
steel

In [ ]:
#Common variables

helix_angle = Q_(30.9749, "degree").to_base_units().m
pr_ang = Q_(20, "degree").to_base_units().m

damping_factor = 1

### Modelagem Rotor de Baixa Velocidade

Modelagem Eixo de Engrenamento - Baixa velocidade

In [ ]:
# Creating a list of low speed gear shaft finite elements

steel_ls = rs.Material(name="AISI_4140", rho=7850 * 0.992887, E=2e11, Poisson=0.3)

L_aux   = np.array([0.03056235, 0.06356968, 0.0599022 , 0.0403423 , 0.02811736,
       0.04645477, 0.04767726, 0.0207824 , 0.0391198 , 0.03789731,
       0.02444988, 0.02444988, 0.0391198 , 0.03789731, 0.02200489,
       0.04767726, 0.04767726, 0.02689487, 0.03300733, 0.01100244,
       0.09413203, 0.09168704, 0.02322738, 0.00244499, 0.085574572])

o_d_aux = np.array([0.18072289, 0.08433735, 0.08433735, 0.19036145, 0.19036145,
       0.19036145, 0.19036145, 0.19036145, 0.78072289, 0.78072289,
       0.76144578, 0.76144578, 0.78072289, 0.78072289, 0.19036145,
       0.19036145, 0.19036145, 0.19277108, 0.19518072, 0.21686747,
       0.19036145, 0.19277108, 0.38795181, 0.29879518, 0.195180723
       ])

L = L_aux[::-1]
o_d = o_d_aux[::-1]

i_d = np.zeros_like(L)

N = len(L)

shaft_gear_ls = [
    rs.ShaftElement(
        L=L[i],
        idl=i_d[i],
        odl=o_d[i],
        material=steel_ls,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(N)
]

# del L, i_d, o_d, N, L_aux, o_d_aux

In [ ]:
# Creating Gear low Speed

n      = [14]
width  = np.array([156.01])*1e-3
i_d    = np.array([189.9])*1e-3
pitch_diameter = np.array([788.44])*1e-3
pressure_angle = np.array([pr_ang])
n_teeth = np.array([169])
helix = np.array([helix_angle])

N = len(n)

# gear_low_speed = [
#     rs.GearElementTVMS.from_geometry(
#         n=n[i],
#         material=steel,
#         width=width[i],
#         i_d=i_d[i],
#         n_teeth=n_teeth[i],
#         o_d=pitch_diameter[i],
#         pr_angle=pressure_angle[i],
#         helix_angle=helix[i],
#     )
#     for i in range(N)
# ]


gear_low_speed = [
    rs.GearElementTVMS(
        n=n[i],
        material=steel,
        width=width[i],
        bore_diameter=i_d[i],
        module=pitch_diameter[i]/n_teeth[i],
        n_teeth=n_teeth[i],
        pr_angle=pressure_angle[i],
        helix_angle = helix[i],
    )
    for i in range(N)
]

# del n, width, i_d, pitch_diameter, pressure_angle, N

In [ ]:
speed_low_blind = np.array([410, 820, 1231, 1641, 2051, 2461, 2871, 3282])*np.pi/30
kxx_low_blind = np.array([1.26e+08, 1.41e+08, 1.69e+08, 1.82e+08, 1.99e+08, 2.16e+08, 2.30e+08, 2.41e+08])
kxy_low_blind = np.array([4.33e+08, 3.27e+08, 3.00e+08, 2.85e+08, 2.81e+08, 2.82e+08, 2.85e+08, 2.86e+08])
kyx_low_blind = np.array([-4.98e+08, -5.77e+08, -6.53e+08, -6.86e+08, -7.25e+08, -7.62e+08, -7.95e+08, -8.22e+08])
kyy_low_blind = np.array([1.90e+09, 1.40e+09, 1.20e+09, 1.10e+09, 1.05e+09, 1.01e+09, 9.87e+08, 9.78e+08])
cxx_low_blind = np.array([3.04e+06, 1.93e+06, 1.59e+06, 1.37e+06, 1.25e+06, 1.18e+06, 1.11e+06, 1.04e+06])*damping_factor
cxy_low_blind = np.array([4.22e+06, 9.47e+05, 2.20e+05, -9.41e+04, -2.24e+05, -2.99e+05, -3.35e+05, -3.51e+05])*damping_factor
cyx_low_blind = np.array([8.52e+05, -4.68e+05, -6.70e+05, -6.04e+05, -5.79e+05, -5.56e+05, -5.25e+05, -4.77e+05])*damping_factor
cyy_low_blind = np.array([5.13e+07, 2.30e+07, 1.48e+07, 1.11e+07, 8.88e+06, 7.49e+06, 6.49e+06, 5.75e+06])*damping_factor

speed_low_ext = np.array([410, 820, 1231, 1641, 2051, 2461, 2871, 3282])*np.pi/30
kxx_low_ext = np.array([1.70e+08, 1.93e+08, 2.03e+08, 2.24e+08, 2.43e+08, 2.62e+08, 2.81e+08, 2.96e+08])
kxy_low_ext = np.array([4.37e+08, 3.32e+08, 2.93e+08, 2.81e+08, 2.76e+08, 2.80e+08, 2.84e+08, 2.88e+08])
kyx_low_ext = np.array([-7.33e+08, -7.69e+08, -7.75e+08, -8.14e+08, -8.47e+08, -8.82e+08, -9.16e+08, -9.44e+08])
kyy_low_ext = np.array([2.38e+09, 1.72e+09, 1.48e+09, 1.34e+09, 1.26e+09, 1.20e+09, 1.17e+09, 1.14e+09])
cxx_low_ext = np.array([3.27e+06, 2.20e+06, 1.72e+06, 1.52e+06, 1.37e+06, 1.27e+06, 1.21e+06, 1.15e+06])*damping_factor
cxy_low_ext = np.array([2.71e+06, 1.18e+06, -4.18e+05, -5.37e+05, -5.75e+05, -5.83e+05, -5.92e+05, -5.77e+05])*damping_factor
cyx_low_ext = np.array([-1.53e+06, -1.71e+06, -1.24e+06, -1.09e+06, -9.43e+05, -8.49e+05, -7.92e+05, -7.23e+05])*damping_factor
cyy_low_ext = np.array([6.18e+07, 2.73e+07, 1.74e+07, 1.28e+07, 1.02e+07, 8.48e+06, 7.33e+06, 6.45e+06])*damping_factor

In [ ]:
# Creating Bearing Low Speed Rotor

n   = [9, 19]
tag = ['Gearbox A', 'Gearbox B']
kxx = [kxx_low_ext, kxx_low_blind]
kyy = [kyy_low_ext, kyy_low_blind]
kxy = [kxy_low_ext, kxy_low_blind]
kyx = [kyx_low_ext, kyx_low_blind]
cxx = [cxx_low_ext, cxx_low_blind]
cyy = [cyy_low_ext, cyy_low_blind]
cxy = [cxy_low_ext, cxy_low_blind]
cyx = [cyx_low_ext, cyx_low_blind]
freq = [speed_low_ext, speed_low_blind]
N = len(n)

bearing_ls_gearbox = [
    rs.BearingElement(
        n=n[i],
        kxx = kxx[i],
        kyy = kyy[i],
        kxy = kxy[i],
        kyx = kyx[i],
        cxx = cxx[i],
        cyy = cyy[i],
        cxy = cxy[i],
        cyx = cyx[i],
        tag = tag[i],
        frequency = freq[i]
    )
    for i in range(N)
]

# del n, kxx, kyy, kxy, kyx, cxx, cyy, cxy, cyx, N, freq, speed_low_ext, speed_low_blind
# del kxx_low_blind , kxx_low_ext, kyx_low_blind , kyx_low_ext, kyy_low_blind , kyy_low_ext, kxy_low_blind , kxy_low_ext
# del cxx_low_blind , cxx_low_ext, cyx_low_blind , cyx_low_ext, cyy_low_blind , cyy_low_ext, cxy_low_blind , cxy_low_ext


In [ ]:
rotor_ls_gearbox = rs.Rotor(shaft_gear_ls, gear_low_speed, bearing_ls_gearbox)

In [ ]:
rotor_ls_gearbox.plot_rotor(nodes=1)

Ciração do Rotor de Baixa velocidade

In [ ]:
rotor_ls = concatenate_rotor([rotor_ls_gearbox])

# del motor, coupling_ls, rotor_ls_gearbox

### Modelagem do Rotor de Alta Velocidade 

Modelagem Eixo de Engrenamento - Alta velocidade

In [ ]:
# Creating a list of high speed gear shaft finite elements

steel_hs = rs.Material(name="AISI_4140", rho=7850 * 1.016, E=2e11, Poisson=0.3)

L_aux   = np.array([0.03870968, 0.02822581, 0.02580645, 0.03387097, 0.03629032,
       0.03870968, 0.03870968, 0.01935484, 0.01048387, 0.03951613,
       0.03870968, 0.02419355, 0.0233871 , 0.03870968, 0.03870968,
       0.01048387, 0.02903226, 0.03870968, 0.04112903, 0.025     ,
       0.01451613, 0.00967742, 0.03306452, 0.03145161, 0.01532258,
       0.03064516, 0.04112903, 0.01209677, 0.00887097])

o_d_aux = np.array([0.0899, 0.07377049, 0.07377049, 0.08032787, 0.08032787,
       0.08032787, 0.08032787, 0.08032787, 0.10655738, 0.12459016,
       0.12459016, 0.10819672, 0.10655738, 0.12459016, 0.12459016,
       0.10655738, 0.08032787, 0.07868852, 0.08032787, 0.07868852,
       0.07868852, 0.10655738, 0.07868852, 0.07868852, 0.05737705,
       0.05737705, 0.05737705, 0.08032787, 0.08032787])

L = L_aux[::-1]
o_d = o_d_aux[::-1]

i_d = np.zeros_like(L)

N = len(L)
shaft_gear_hs = [
    rs.ShaftElement(
        L=L[i],
        idl=i_d[i],
        odl=o_d[i],
        material=steel_hs,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(N)
]

# del L, i_d, o_d, N, L_aux, o_d_aux

In [ ]:
# Creating Gear High Speed
n      = [17]
width  = np.array([156.01])*1e-3
i_d    = np.array([60])*1e-3
pitch_diameter = np.array([125.96])*1e-3
pressure_angle = np.array([pr_ang])
n_teeth = np.array([27])
helix = np.array([helix_angle])

N = len(n)

# gear_high_speed = [
#     rs.GearElement.from_geometry(
#         n=n[i],
#         material=steel,
#         width=width[i],
#         i_d=i_d[i],
#         o_d=pitch_diameter[i],
#         n_teeth=n_teeth[i],
#         pr_angle=pressure_angle[i],
#         helix_angle=helix[i],
#     )
#     for i in range(N)
# ]

gear_high_speed = [
    rs.GearElementTVMS(
        n=n[i],
        material=steel,
        width=width[i],
        bore_diameter=i_d[i],
        module=pitch_diameter[i]/n_teeth[i],
        n_teeth=n_teeth[i],
        pr_angle=pressure_angle[i],
        helix_angle = helix[i],
    )
    for i in range(N)
]

# del n, width, i_d, pitch_diameter, pressure_angle, N

In [ ]:
speed_hs_blind = np.array([2567, 5134, 7702, 10269, 12836, 15403, 17970, 20538])*np.pi/30
kxx_hs_blind = np.array([8.16e+08, 6.55e+08, 6.24e+08, 6.22e+08, 6.29e+08, 6.40e+08, 6.54e+08, 6.68e+08])
kyx_hs_blind = np.array([-1.10e+09, -8.37e+08, -7.70e+08, -7.52e+08, -7.53e+08, -7.64e+08, -7.79e+08, -7.96e+08])
kxy_hs_blind = np.array([-2.75e+08, -5.44e+07, 4.11e+07, 8.72e+07, 1.15e+08, 1.27e+08, 1.32e+08, 1.35e+08])
kyy_hs_blind = np.array([7.92e+08, 4.72e+08, 3.70e+08, 3.27e+08, 3.05e+08, 2.92e+08, 2.83e+08, 2.77e+08])
cxx_hs_blind = np.array([3.52e+06, 1.65e+06, 1.12e+06, 8.53e+05, 6.94e+05, 5.77e+05, 4.92e+05, 4.29e+05])*damping_factor
cxy_hs_blind = np.array([-3.37e+06, -1.41e+06, -9.05e+05, -6.65e+05, -5.27e+05, -4.35e+05, -3.70e+05, -3.22e+05])*damping_factor
cyx_hs_blind = np.array([-3.95e+06, -1.61e+06, -9.80e+05, -6.94e+05, -5.35e+05, -4.36e+05, -3.69e+05, -3.21e+05])*damping_factor
cyy_hs_blind = np.array([4.77e+06, 1.98e+06, 1.26e+06, 9.35e+05, 7.54e+05, 6.39e+05, 5.58e+05, 4.99e+05])*damping_factor

speed_hs_ext = np.array([2567, 5134, 7702, 10269, 12836, 15403, 17970, 20538])*np.pi/30
kxx_hs_ext = np.array([7.97e+08, 6.49e+08, 6.17e+08, 6.14e+08, 6.21e+08, 6.33e+08, 6.46e+08, 6.61e+08])
kxy_hs_ext = np.array([-2.72e+08, -3.35e+07, 4.72e+07, 8.99e+07, 1.13e+08, 1.23e+08, 1.26e+08, 1.29e+08])
kyx_hs_ext = np.array([-1.06e+09, -8.12e+08, -7.49e+08, -7.33e+08, -7.35e+08, -7.46e+08, -7.62e+08, -7.81e+08])
kyy_hs_ext = np.array([7.57e+08, 4.43e+08, 3.53e+08, 3.14e+08, 2.94e+08, 2.82e+08, 2.74e+08, 2.69e+08])
cxx_hs_ext = np.array([3.39e+06, 1.67e+06, 1.11e+06, 8.42e+05, 6.76e+05, 5.62e+05, 4.78e+05, 4.18e+05])*damping_factor
cxy_hs_ext = np.array([-3.20e+06, -1.41e+06, -8.83e+05, -6.49e+05, -5.11e+05, -4.22e+05, -3.59e+05, -3.13e+05])*damping_factor
cyx_hs_ext = np.array([-3.78e+06, -1.58e+06, -9.49e+05, -6.71e+05, -5.18e+05, -4.22e+05, -3.58e+05, -3.12e+05])*damping_factor
cyy_hs_ext = np.array([4.53e+06, 1.92e+06, 1.21e+06, 9.05e+05, 7.31e+05, 6.21e+05, 5.43e+05, 4.86e+05])*damping_factor

In [ ]:
# Creating Bearing high Speed Rotor

n   = [11, 23]
kxx = [kxx_hs_ext, kxx_hs_blind]
kyy = [kyy_hs_ext, kyy_hs_blind]
kxy = [kxy_hs_ext, kxy_hs_blind]
kyx = [kyx_hs_ext, kyx_hs_blind]
cxx = [cxx_hs_ext, cxx_hs_blind]
cyy = [cyy_hs_ext, cyy_hs_blind]
cxy = [cxy_hs_ext, cxy_hs_blind]
cyx = [cyx_hs_ext, cyx_hs_blind]
freq = [speed_hs_ext, speed_hs_blind]
N = len(n)

bearing_hs_gearbox = [
    rs.BearingElement(
        n=n[i],
        kxx = kxx[i],
        kyy = kyy[i],
        kxy = kxy[i],
        kyx = kyx[i],
        cxx = cxx[i],
        cyy = cyy[i],
        cxy = cxy[i],
        cyx = cyx[i],
        frequency = freq[i]
    )
    for i in range(N)
]

# del n, kxx, kyy, kxy, kyx, cxx, cyy, cxy, cyx, N, freq, speed_hs_ext, speed_hs_blind
# del kxx_hs_blind , kxx_hs_ext, kyx_hs_blind , kyx_hs_ext, kyy_hs_blind , kyy_hs_ext, kxy_hs_blind , kxy_hs_ext
# del cxx_hs_blind , cxx_hs_ext, cyx_hs_blind , cyx_hs_ext, cyy_hs_blind , cyy_hs_ext, cxy_hs_blind , cxy_hs_ext



In [ ]:
rotor_hs_gearbox = rs.Rotor(shaft_gear_hs, gear_high_speed, bearing_hs_gearbox)

# del shaft_gear_hs, gear_high_speed, bearing_hs_gearbox

In [ ]:
rotor_hs_gearbox.plot_rotor()

Configurações do Rotor de Alta Velocidade

In [ ]:
rotor_hs = concatenate_rotor([rotor_hs_gearbox])

# del rotor_hs_gearbox, coupling_hs, compressor

### Determinação do Multi Rotor

In [ ]:
gear_coupling_stifness = 2e9 #de acordo com a análise de variação dos mod

n_gear_ls = next(e.n for e in rotor_ls.disk_elements if isinstance(e, rs.GearElement))
n_gear_hs = next(e.n for e in rotor_hs.disk_elements if isinstance(e, rs.GearElement))
coupled_nodes = (n_gear_hs,n_gear_ls)

multirotor = rs.MultiRotor(
    rotor_hs, 
    rotor_ls, 
    coupled_nodes=coupled_nodes, 
    # gear_mesh_stiffness=gear_coupling_stifness,
    update_mesh_stiffness=True,
    # square_varying_stiffness = True,
    # square_stiffness_amplitude_ratio = 0.225,
    position="above",
    orientation_angle=0 * np.pi/180,
    )
# del n_gear_hs, n_gear_ls, gear_ratio, gear_coupling_stifness, coupled_nodes

In [ ]:
multirotor.mesh.stiffness/1e9

In [ ]:
multirotor.mesh.contact_ratio

In [ ]:
multirotor.plot_rotor(nodes=999)

In [ ]:
multirotor.mesh.plot_stiffness_profile(n_mesh_period=2)

In [ ]:
speed_driving_gear = Q_(1200,"rpm").to_base_units().m
mag1 = Q_(100e6,"g*mm").to_base_units().m
mag2 = Q_(5000e6,"g*mm").to_base_units().m
ph1 = Q_(0,"deg").to_base_units().m
ph2 = Q_(0,"deg").to_base_units().m

In [ ]:
unb_node = [e.n for e in multirotor.disk_elements if isinstance(e, rs.GearElement)]
unb_magnitude = [mag1, mag2]
unb_phase = [ph1, ph2]

In [ ]:
backlash = Backlash(multirotor,
                    speed_driving_gear,
                    b0=0*50e-6,
                    error_amp = 0,
                    gear_mesh_stiffness = None,
                    num_points_cicle=10000, 
                    n_cicles=15, 
                    cut_cicles=5,
                    use_multirotor_coupling_stiffness = False,
                    RHS = False,
                    )

In [ ]:
unb_node

In [ ]:
backlash.time[1] - backlash.time[0]

In [ ]:
backlash.time[-1]

In [ ]:

F = np.zeros((len(backlash.time), multirotor.ndof))

T = 10 * np.sin(backlash.time)

# component on direction x
F[:, unb_node[0] * multirotor.number_dof + 5] = T
# component on direction y
F[:, unb_node[1] * multirotor.number_dof + 5] = -T

In [ ]:
F.shape

In [ ]:
break

In [ ]:
backlash.run_dynamic_backlash(unb_node,
                              unb_magnitude,
                              unb_phase,
                            #   add_force = F,
                            # integration_method="piriri"
                              )

In [ ]:
backlash.backlash_results["f"]

In [ ]:
backlash.time

In [ ]:

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["Fm"],
        mode="lines",
        name="Fm(K, δ, bt)"
    )
)

fig.update_layout(
    xaxis_title="Time",
    yaxis_title="Fm",
    template="plotly_white"
)

fig.show()

# fig.update_layout(
#     title="Delta vs Time",
#     xaxis=dict(showgrid=True),
#     yaxis=dict(showgrid=True)
# )



In [ ]:
probe1 = Probe(unb_node[0],0)
probe2 = Probe(unb_node[1],0)

backlash.time_response.plot_1d([probe1,probe2])

In [ ]:
backlash.time_response.plot_2d(unb_node[0])

In [ ]:
backlash.time_response.plot_2d(unb_node[1])

In [ ]:
backlash.time_response.plot_dfft([probe1,probe2], yaxis=dict(type="linear"), frequency_units="rpm")

In [ ]:
# backlash.time_response.plot_3d()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["d"],
        mode="lines",
        name="d(t)"
    )
)

d0_vec = np.ones_like(backlash.time)
d0 = d0_vec*(multirotor.gear_1.pitch_diameter + multirotor.gear_2.pitch_diameter)/2 
fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=d0,
        mode="lines",
        name="d0"
    )
)

RA_vec = np.ones_like(backlash.time)

RA = RA_vec*(multirotor.gear_1.radii_dict["addendum"] + multirotor.gear_2.radii_dict["addendum"])

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=RA,
        mode="lines",
        name="Ra1+Ra2"
    )
)

Rf_vec = np.ones_like(backlash.time)

Rf = Rf_vec*(multirotor.gear_1.radii_dict["root"] + multirotor.gear_2.radii_dict["root"])

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=Rf,
        mode="lines",
        name="Rf1+Rf2"
    )
)

fig.update_layout(
    xaxis_title="Time",
    yaxis_title="d",
    template="plotly_white"
)

fig.show()

# fig.update_layout(
#     title="Delta vs Time",
#     xaxis=dict(showgrid=True),
#     yaxis=dict(showgrid=True)
# )

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.backlash_results["x1"],
        y=backlash.backlash_results["y1"],
        mode="lines",
        name="R1"
    )
)

fig.add_trace(
    go.Scatter(
        x=backlash.backlash_results["x2"],
        y=backlash.backlash_results["y2"],
        mode="lines",
        name="R2"
    )
)

fig.update_layout(
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white"
)

fig.show()

# fig.update_layout(
#     title="Delta vs Time",
#     xaxis=dict(showgrid=True),
#     yaxis=dict(showgrid=True)
# )

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["alfa"]*180/np.pi,
        mode="lines",
        name="alfa(t)"
    )
)

alfa0_vec = np.ones_like(backlash.time)
alfa0 = alfa0_vec*multirotor.gear_1.pr_angle*180/np.pi
fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=alfa0,
        mode="lines",
        name="alfa0"
    )
)

fig.update_layout(
    xaxis_title="Time",
    yaxis_title="alpha",
    template="plotly_white"
)

fig.show()

# fig.update_layout(
#     title="Delta vs Time",
#     xaxis=dict(showgrid=True),
#     yaxis=dict(showgrid=True)
# )

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["beta"]*180/np.pi,
        mode="lines",
        name="beta(t)"
    )
)


fig.update_layout(
    xaxis_title="Time",
    yaxis_title="beta",
    template="plotly_white"
)

beta0_vec = np.ones_like(backlash.time)
beta0 = beta0_vec*multirotor.orientation_angle*180/np.pi
fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=beta0,
        mode="lines",
        name="beta0"
    )
)

fig.show()

# fig.update_layout(
#     title="Delta vs Time",
#     xaxis=dict(showgrid=True),
#     yaxis=dict(showgrid=True)
# )

In [ ]:
# import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["delta"],
        mode="lines",
        name="δ(t)"
    )
)

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["bt"],
        mode="lines",
        name="bt"
    )
)

fig.update_layout(
    xaxis_title="Time",
    yaxis_title="Delta",
    template="plotly_white"
)

fig.show()

# fig.update_layout(
#     title="Delta vs Time",
#     xaxis=dict(showgrid=True),
#     yaxis=dict(showgrid=True)
# )



In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["contact_ratio"],
        mode="lines",
        name="contact_ratio"
    )
)

# fig.add_trace(
#     go.Scatter(
#         x=t,
#         y=K_time,
#         mode="lines",
#         name="k"
#     )
# )

fig.update_layout(
    xaxis_title="time",
    yaxis_title="contact_ratio",
    template="plotly_white"
)

fig.show()

In [ ]:

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["f"],
        mode="lines",
        name="f(δ,bt)"
    )
)

fig.update_layout(
    xaxis_title="Time",
    yaxis_title="f",
    template="plotly_white"
)

fig.show()

# fig.update_layout(
#     title="Delta vs Time",
#     xaxis=dict(showgrid=True),
#     yaxis=dict(showgrid=True)
# )



In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["K_time"],
        mode="lines",
        name="K_time"
    )
)

# fig.add_trace(
#     go.Scatter(
#         x=t,
#         y=K_time,
#         mode="lines",
#         name="k"
#     )
# )

fig.update_layout(
    xaxis_title="time",
    yaxis_title="K_time",
    template="plotly_white"
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=backlash.time,
        y=backlash.backlash_results["Fm"],
        mode="lines",
        name="f*k_time"
    )
)

# fig.add_trace(
#     go.Scatter(
#         x=t,
#         y=K_time,
#         mode="lines",
#         name="k"
#     )
# )

fig.update_layout(
    xaxis_title="time",
    yaxis_title="Fm",
    template="plotly_white"
)

fig.show()

In [ ]:
# import numpy as np
# import plotly.graph_objects as go


def dfft(
    x,
    t,
    freq_unit="Hz",
    remove_mean=True,
    window="hann"
):
    x = np.asarray(x)
    t = np.asarray(t)

    if len(x) != len(t):
        raise ValueError("x e t devem ter o mesmo comprimento")

    dt = t[1] - t[0]

    if not np.allclose(np.diff(t), dt):
        raise ValueError("O vetor de tempo deve ter amostragem uniforme")

    N = len(x)

    # Remove média (remove componente DC)
    if remove_mean:
        x = x - np.mean(x)

    # Aplicar janela
    if window == "hann":
        w = np.hanning(N)
        x = x * w
        correction = 1 / np.mean(w)
    else:
        correction = 1.0

    # FFT real (somente parte positiva)
    X = np.fft.rfft(x)
    freq = np.fft.rfftfreq(N, d=dt)

    amplitude = (2.0 / N) * np.abs(X) * correction

    # Conversão de unidade de frequência
    if freq_unit == "rad/s":
        freq = 2 * np.pi * freq
        freq_label = "rad/s"
    elif freq_unit == "rpm":
        freq = freq * 60
        freq_label = "rpm"
    else:
        freq_label = "Hz"

    return freq, amplitude, freq_label


def plot_dfft(
    freq,
    amplitude,
    freq_label,
    title="DFFT",
    xlim=None,
    ylim=None,
    tipo="linear"
):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=freq,
            y=amplitude,
            mode="lines",
            name="Espectro"
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title=f"Frequência [{freq_label}]",
        template="plotly_white",
        width=900,
        height=500,
        yaxis=dict(type=tipo)
    )

    if xlim is not None:
        fig.update_xaxes(range=xlim)

    if ylim is not None:
        fig.update_yaxes(range=ylim)

    fig.show()


In [ ]:
freq, amp, f_label = dfft(
    backlash.backlash_results["Fm"],
    backlash.time,
    freq_unit="rpm",
    remove_mean=False,
    # window=None,
)

plot_dfft(
    freq,
    amp,
    f_label,
    # xlim=[0, 500],
    title="Espectro da Força",
    # tipo="log"
)